In [ ]:
# ============================================
# 03_catboost_train_short.ipynb
# Train CatBoost model for ShortSuccess
# ============================================

import os
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from catboost.utils import convert_to_onnx
from skl2onnx.common.data_types import FloatTensorType

# ---------------- CONFIG ----------------
DATA_PATH = r"C:\Trading\Projects\ES_AI_Project\data\labels\labeled.csv"
OUTPUT_DIR = r"C:\Trading\Projects\ES_AI_Project\models\catboost"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET = "ShortSuccess"

DROP_COLS = [
    "Time", "IsTradeBar", "IsLong", "IsShort",
    "BarsSinceEntry", "FutureClose", "FutureRet",
    "LongSuccess", "ShortSuccess"
]

# ---------------- LOAD DATA ----------------
df = pd.read_csv(DATA_PATH)

# ---------------- FEATURES ----------------
feature_cols = [c for c in df.columns if c not in DROP_COLS]

X = df[feature_cols]
y = df[TARGET].astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_pool = Pool(X_train, y_train)
val_pool = Pool(X_val, y_val)

# ---------------- MODEL ----------------
model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    depth=6,
    learning_rate=0.05,
    iterations=500,
    random_seed=42,
    verbose=100,
    od_type="Iter",
    od_wait=50,
)

model.fit(train_pool, eval_set=val_pool, use_best_model=True)

# ---------------- SAVE CATBOOST MODEL (.cbm) ----------------
cbm_path = os.path.join(OUTPUT_DIR, "catboost_ShortSuccess.cbm")
model.save_model(cbm_path)

# ---------------- CORRECT ONNX EXPORT (TENSOR OUTPUT) ----------------
onnx_path = os.path.join(OUTPUT_DIR, "catboost_ShortSuccess.onnx")

onnx_model = convert_to_onnx(
    model,
    initial_types=[("features", FloatTensorType([None, len(feature_cols)]))],
    target_opset=13,
    export_parameters={
        "onnx_export_class_labels": False,
        "onnx_export_probabilities": True
    }
)

with open(onnx_path, "wb") as f:
    f.write(onnx_model.SerializeToString())

print("Saved ONNX model:", onnx_path)

# ---------------- SAVE FEATURE LIST ----------------
feat_path = os.path.join(OUTPUT_DIR, "features_ShortSuccess.txt")
with open(feat_path, "w") as f:
    for c in feature_cols:
        if c != TARGET:
            f.write(c + "\n")

print("Short model training complete.")


ImportError: cannot import name 'convert_to_onnx' from 'catboost.utils' (c:\Trading\Projects\ES_AI_Project\es_env310\lib\site-packages\catboost\utils.py)